In [ ]:
!pip install requests beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Base URL of the GDPR website
base_url = "https://gdpr-info.eu/"

# Send a GET request to the main URL
response = requests.get(base_url)

# Check if the request was successful
if response.status_code == 200:
    # Parse the HTML content of the main page
    soup = BeautifulSoup(response.content, "html.parser")

    # Lists to store article names, URLs, and article contents
    article_names = []
    article_urls = []
    article_contents = []

    # Find all anchor tags within the navigation or relevant sections
    links = soup.find_all('a', href=True)
    for link in links:
        link_text = link.get_text(strip=True)
        link_url = link['href']

        # Filter links that match the specific format "http://gdpr-info.eu/art-<number>-gdpr/"
        if link_url.startswith("http://gdpr-info.eu/art-") and link_url.endswith("-gdpr/"):
            # Construct the full URL if the link is relative
            full_url = link_url if link_url.startswith("http") else base_url + link_url.lstrip('/')

            # Fetch the content of each article link
            article_response = requests.get(full_url)

            if article_response.status_code == 200:
                article_soup = BeautifulSoup(article_response.content, "html.parser")

                # Extract content within the div tag with class "entry-content"
                entry_content_div = article_soup.find('div', class_='entry-content')

                # Check if the div with the specified class was found
                if entry_content_div:
                    # Extract text only from `li` and `p` tags within `entry-content`
                    list_items = entry_content_div.find_all(['li', 'p'])
                    full_text = "\n".join([item.get_text(strip=True) for item in list_items])
                else:
                    full_text = "No 'entry-content' div found or no content in specified tags."

                # Store the article name, URL, and content
                article_names.append(link_text)
                article_urls.append(full_url)
                article_contents.append(full_text)

                # Print the content for verification
                print(f"Article Name: {link_text}\nURL: {full_url}\nContent: {full_text}\n")
            else:
                print(f"Failed to retrieve article content at {full_url}")
                article_contents.append("")

    # Create a DataFrame
    df = pd.DataFrame({
        'Article Name': article_names,
        'URL': article_urls,
        'Content': article_contents
    })

    # Display the DataFrame
    print(df)

    # Optionally, save the DataFrame to a CSV file
    # df.to_csv('gdpr_articles_filtered_with_li_and_p_content.csv', index=False)

else:
    print(f"Failed to retrieve content from the website. Status code: {response.status_code}")


Article Name: 1
URL: http://gdpr-info.eu/art-1-gdpr/
Content: This Regulation lays down rules relating to the protection of natural persons with regard to the processing of personal data and rules relating to the free movement of personal data.
This Regulation protects fundamental rights and freedoms of natural persons and in particular their right to the protection of personal data.
The free movement of personal data within the Union shall be neither restricted nor prohibited for reasons connected with the protection of natural persons with regard to the processing of personal data.

Article Name: 2
URL: http://gdpr-info.eu/art-2-gdpr/
Content: This Regulation applies to the processing of personal data wholly or partly by automated means and to the processing other than by automated means of personal data which form part of a filing system or are intended to form part of a filing system.
This Regulation does not apply to the processing of personal data:in the course of an activity whi

In [ ]:
df.iloc[11]['Content']

'1The controller shall take appropriate measures to provide any information referred to inArticles 13and14and any communication underArticles 15to22and34relating to processing to the data subject in a concise, transparent, intelligible and easily accessible form, using clear and plain language, in particular for any information addressed specifically to a child.2The information shall be provided in writing, or by other means, including, where appropriate, by electronic means.3When requested by the data subject, the information may be provided orally, provided that the identity of the data subject is proven by other means.\n1The controller shall facilitate the exercise of data subject rights underArticles 15to22.2In the cases referred to inArticle 11(2), the controller shall not refuse to act on the request of the data subject for exercising his or her rights underArticles 15to22, unless the controller demonstrates that it is not in a position to identify the data subject.\n1The control

In [ ]:
df.iloc[11]['URL']

'http://gdpr-info.eu/art-12-gdpr/'

In [ ]:
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download required resources if not already present
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

stop_words = set(stopwords.words('english'))
punctuations = string.punctuation
lemmatizer = WordNetLemmatizer()

def process_text(text):
  # Remove punctuations and convert to lowercase
  text = text.translate(str.maketrans('', '', punctuations)).lower()
  # Remove numbers
  text = ''.join([i for i in text if not i.isdigit()])
  # Remove stop words and lemmatize
  text = ' '.join([lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words])
  return text

df['Processed Content'] = df['Content'].apply(process_text)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
df2.to_csv('regulatory_preprocessed_papers_popi.csv', index=False, escapechar='\\')

In [ ]:
df2 = pd.read_csv('regulatory_preprocessed_papers_popi.csv', escapechar='\\')

In [ ]:
df2['Content'][69]

'A data subject who is a subscriber to a printed or electronic directory of subscribers available to the public or obtainable through directory enquiry services, in which his, her or its personal information is included, must be informed, free of charge and before the information is included in the directory—about the purpose of the directory; andabout any further uses to which the directory may possibly be put, based on search functions embedded in electronic versions of the directory.\nabout the purpose of the directory; and\nabout any further uses to which the directory may possibly be put, based on search functions embedded in electronic versions of the directory.\nA data subject must be given a reasonable opportunity to object, free of charge and in a manner free of unnecessary formality, to such use of his, her or its personal information or to request verification, confirmation or withdrawal of such information if the data subject has not initially refused such use.\nSubsections

In [ ]:
import csv
# df.to_csv('regulatory_preprocessed_papers_gdpr.csv', index=False, escapechar='\\', quoting=csv.QUOTE_NONE)

In [ ]:
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')

def tokenize_text(text):
  # Check if the input is a string before tokenizing
  if isinstance(text, str):
    tokens = word_tokenize(text)
    return tokens
  else:
    # Handle non-string values, e.g., return an empty list or a placeholder
    return []  # Or return a placeholder like ['<NON_STRING>']

df['Tokens'] = df['Processed Content'].apply(tokenize_text)

df['Tokens'][69]


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


['general',
 'data',
 'protection',
 'regulation',
 'gdpr',
 'final',
 'text',
 'gdpr',
 'including',
 'recital',
 'gdpr',
 'recital',
 'key',
 'issue',
 'ai',
 'act',
 'gdpr',
 'recital',
 'key',
 'issue',
 'ai',
 'act',
 'gdpr',
 'chapter',
 'art',
 '–',
 'general',
 'provisionsart',
 'subjectmatter',
 'objectivesart',
 'material',
 'scopeart',
 'territorial',
 'scopeart',
 'definition',
 'art',
 'subjectmatter',
 'objective',
 'art',
 'material',
 'scope',
 'art',
 'territorial',
 'scope',
 'art',
 'definition',
 'chapter',
 'art',
 '–',
 'principlesart',
 'principle',
 'relating',
 'processing',
 'personal',
 'dataart',
 'lawfulness',
 'processingart',
 'condition',
 'consentart',
 'condition',
 'applicable',
 'child',
 '’',
 's',
 'consent',
 'relation',
 'information',
 'society',
 'servicesart',
 'processing',
 'special',
 'category',
 'personal',
 'dataart',
 'processing',
 'personal',
 'data',
 'relating',
 'criminal',
 'conviction',
 'offencesart',
 'processing',
 'require',


## POPI Regulatory Document

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Base URL of the GDPR website
base_url = "https://popia.co.za/"

# Send a GET request to the main URL
response = requests.get(base_url)

# Check if the request was successful
if response.status_code == 200:
    # Parse the HTML content of the main page
    soup = BeautifulSoup(response.content, "html.parser")

    # Lists to store article names, URLs, and article contents
    article_names = []
    article_urls = []
    article_contents = []

    # Find all anchor tags within the navigation or relevant sections
    links = soup.find_all('a', href=True)
    for link in links:
        link_text = link.get_text(strip=True)
        link_url = link['href']

        # Filter links that match the specific format "http://gdpr-info.eu/art-<number>-gdpr/"
        if link_url.startswith("https://popia.co.za/section-") and link_url.endswith("/"):
            # Construct the full URL if the link is relative
            full_url = link_url if link_url.startswith("http") else base_url + link_url.lstrip('/')

            # Fetch the content of each article link
            article_response = requests.get(full_url)

            if article_response.status_code == 200:
                article_soup = BeautifulSoup(article_response.content, "html.parser")

                # Extract content within the div tag with class "entry-content"
                entry_content_div = article_soup.find('div', class_='entry-content')

                # Check if the div with the specified class was found
                if entry_content_div:
                    # Extract text only from `li` and `p` tags within `entry-content`
                    list_items = entry_content_div.find_all(['li', 'p'])
                    full_text = "\n".join([item.get_text(strip=True) for item in list_items])
                else:
                    full_text = "No 'entry-content' div found or no content in specified tags."

                # Store the article name, URL, and content
                article_names.append(link_text)
                article_urls.append(full_url)
                article_contents.append(full_text)

                # Print the content for verification
                print(f"Article Name: {link_text}\nURL: {full_url}\nContent: {full_text}\n")
            else:
                print(f"Failed to retrieve article content at {full_url}")
                article_contents.append("")

    # Create a DataFrame
    df2 = pd.DataFrame({
        'Article Name': article_names,
        'URL': article_urls,
        'Content': article_contents
    })

    # Display the DataFrame
    print(df2)

    # Optionally, save the DataFrame to a CSV file
    # df.to_csv('gdpr_articles_filtered_with_li_and_p_content.csv', index=False)

else:
    print(f"Failed to retrieve content from the website. Status code: {response.status_code}")


Article Name: Section 1 Definitions
URL: https://popia.co.za/section-1-definitions/
Content: In this Act, unless the context indicates otherwise —
‘‘biometrics’’ means a technique of personal identification that is based on physical, physiological or behavioural characterisation including blood typing, fingerprinting, DNA analysis, retinal scanning and voice recognition;‘‘child’’ means a natural person under the age of 18 years who is not legally competent, without the assistance of a competent person, to take any action or decision in respect of any matter concerning him- or herself;‘‘code of conduct’’ means acode of conductissued in terms of Chapter 7;‘‘competent person’’ means any person who is legally competent to consent to any action or decision being taken in respect of any matter concerning a child;‘‘consent’’ means any voluntary, specific and informed expression of will in terms of which permission is given for the processing of personal information;‘‘Constitution’’ means the 

In [ ]:
df2.iloc[10]['Content']

'Personal information may only be processed if—the data subject or a competentpersonwhere the data subject is achildconsentsto the processing;processing is necessary to carry out actions for the conclusion or performance of a contract to which the data subject is party;processing complies with an obligation imposed by law on the responsible party;processing protects a legitimate interest of the data subject;processing is necessary for the proper performance of a public law duty by a public body;orprocessing is necessary for pursuing the legitimate interests of the responsible party or of a third party to whom the information is supplied.\nthe data subject or a competentpersonwhere the data subject is achildconsentsto the processing;\nprocessing is necessary to carry out actions for the conclusion or performance of a contract to which the data subject is party;\nprocessing complies with an obligation imposed by law on the responsible party;\nprocessing protects a legitimate interest of 

In [ ]:
df2.iloc[10]['URL']

'https://popia.co.za/section-11-consent-justification-and-objection/'

In [ ]:
len(df)

99

In [ ]:
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download required resources if not already present
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

stop_words = set(stopwords.words('english'))
punctuations = string.punctuation
lemmatizer = WordNetLemmatizer()

def process_text(text):
  # Remove punctuations and convert to lowercase
  text = text.translate(str.maketrans('', '', punctuations)).lower()
  # Remove numbers
  text = ''.join([i for i in text if not i.isdigit()])
  # Remove stop words and lemmatize
  text = ' '.join([lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words])
  return text

df2['Processed Content'] = df2['Content'].apply(process_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
df2['Content'][0]

'POPIA\nProtection of Personal Information Act\nACT Summary and Preamble\nChapter 1 Definitions and PurposeSection 1 DefinitionsSection 2 Purpose of Act\nSection 1 Definitions\nSection 2 Purpose of Act\nChapter 2 Application ProvisionsSection 3 Application and interpretation of ActSection 4 Lawful processing of personal informationSection 5 Rights of data subjectsSection 6 ExclusionsSection 7 Exclusion for journalistic, literary or artistic purposes\nSection 3 Application and interpretation of Act\nSection 4 Lawful processing of personal information\nSection 5 Rights of data subjects\nSection 6 Exclusions\nSection 7 Exclusion for journalistic, literary or artistic purposes\nChapter 3 Conditions for Lawful ProcessingPart A Processing of personal information in generalCondition 1 AccountabilitySection 8 Responsible party to ensure conditions for lawful processingCondition 2 Processing limitationSection 9 Lawfulness of processingSection 10 MinimalitySection 11 Consent, justification and o

In [ ]:
df2['Processed Content'][0]

'popia protection personal information act act summary preamble chapter definition purposesection definitionssection purpose act section definition section purpose act chapter application provisionssection application interpretation actsection lawful processing personal informationsection right data subjectssection exclusionssection exclusion journalistic literary artistic purpose section application interpretation act section lawful processing personal information section right data subject section exclusion section exclusion journalistic literary artistic purpose chapter condition lawful processingpart processing personal information generalcondition accountabilitysection responsible party ensure condition lawful processingcondition processing limitationsection lawfulness processingsection minimalitysection consent justification objectionsection collection directly data subjectcondition purpose specificationsection collection specific purposesection retention restriction recordscondi

In [ ]:
df2 = df2.drop(columns=['Content'])
df2.to_csv('regulatory_preprocessed_papers_popi.csv', index=False, escapechar='\\')

In [ ]:
from nltk.tokenize import word_tokenize
nltk.download('punkt')

def tokenize_text(text):
  tokens = word_tokenize(text)
  return tokens

df2['Tokens'] = df2['Processed Content'].apply(tokenize_text)

df2['Tokens'][69]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


['popia',
 'protection',
 'personal',
 'information',
 'act',
 'act',
 'summary',
 'preamble',
 'chapter',
 'definition',
 'purposesection',
 'definitionssection',
 'purpose',
 'act',
 'section',
 'definition',
 'section',
 'purpose',
 'act',
 'chapter',
 'application',
 'provisionssection',
 'application',
 'interpretation',
 'actsection',
 'lawful',
 'processing',
 'personal',
 'informationsection',
 'right',
 'data',
 'subjectssection',
 'exclusionssection',
 'exclusion',
 'journalistic',
 'literary',
 'artistic',
 'purpose',
 'section',
 'application',
 'interpretation',
 'act',
 'section',
 'lawful',
 'processing',
 'personal',
 'information',
 'section',
 'right',
 'data',
 'subject',
 'section',
 'exclusion',
 'section',
 'exclusion',
 'journalistic',
 'literary',
 'artistic',
 'purpose',
 'chapter',
 'condition',
 'lawful',
 'processingpart',
 'processing',
 'personal',
 'information',
 'generalcondition',
 'accountabilitysection',
 'responsible',
 'party',
 'ensure',
 'conditi

In [ ]:
import openai
import pandas as pd
import json

# Load the data from the uploaded file
file_path = '/content/regulatory_preprocessed_papers_gdpr.csv'
data = pd.read_csv(file_path)
data = data[:10]
# Extract the relevant columns
headings = data['Article Name'].tolist()
texts = data['Processed Content'].tolist()

# Set up OpenAI API Key
openai.api_key = "YOUR_API_KEY"

# Function to extract policy statements and compliance requirements
def extract_policy_and_compliance(text):
    response = openai.ChatCompletion.create(
      model="gpt-3.5-turbo",
      messages=[
            {"role": "system", "content": "Extract policy statements and compliance requirements from regulatory text."},
            {"role": "user", "content": f"Extract policy statements and compliance requirements from the following text:\n{text}"}
        ],
        max_tokens=500,
        n=1,
        stop=None,
        temperature=0.2,
    )

    # Process the response to get structured output
    response_text = response.choices[0].message['content']

    # Parsing to separate policy statements and compliance requirements
    policies = []
    compliance_requirements = []
    lines = response_text.splitlines()
    current_list = None
    for line in lines:
        if "Policies:" in line:
            current_list = policies
        elif "Compliance:" in line:
            current_list = compliance_requirements
        elif line.startswith("- "):
            if current_list is not None:
                current_list.append(line[2:])  # Remove leading "- "

    return policies, compliance_requirements

# Dictionary to store the extracted data
extracted_data = {}

# Loop over the regulatory texts and extract policies and compliance requirements
for heading, text in zip(headings, texts):
    policies, compliance_requirements = extract_policy_and_compliance(text)
    extracted_data[heading] = {
        "policies": policies,
        "compliance_requirements": compliance_requirements
    }

# Save the results to a JSON file
output_file = 'extracted_policies_and_requirements.json'
# os.makedirs(os.path.dirname(output_file), exist_ok=True)
with open(output_file, 'w') as f:
    json.dump(extracted_data, f, indent=4)

print(f"Data saved to {output_file}")




Data saved to extracted_policies_and_requirements.json


In [ ]:
data.head(20)

,Article Name,URL,Processed Content
0,General Data Protection Regulation (GDPR),https://gdpr-info.eu/,general data protection regulation gdpr final ...
1,DSGVO,https://dsgvo-gesetz.de/,datenschutzgrundverordnung dsgvo finaler text ...
2,GDPR,https://gdpr-info.eu/,general data protection regulation gdpr final ...
3,Recitals,https://gdpr-info.eu/recitals/,general data protection regulation gdpr final ...
4,Key Issues,https://gdpr-info.eu/issues/,general data protection regulation gdpr final ...
5,AI Act,https://ai-act-law.eu/?utm_source=gdpr-info.eu,ai act kivo ai act recital annex gdpr ai act r...
6,GDPR,https://gdpr-info.eu/,general data protection regulation gdpr final ...
7,Recitals,https://gdpr-info.eu/recitals/,general data protection regulation gdpr final ...
8,Key Issues,https://gdpr-info.eu/issues/,general data protection regulation gdpr final ...
9,AI Act,https://ai-act-law.eu/?utm_source=gdpr-info.eu,ai act kivo ai act recital annex gdpr ai act r...


In [ ]:
import openai
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Initialize OpenAI API Key
openai.api_key = "YOUR_API_KEY"

# Load GDPR and POPIA DataFrames
# gdpr_df = pd.read_csv("gdpr.csv")
# popi_df = pd.read_csv("popia.csv")

# Combine and Label Data
df['label'] = 'GDPR'
df2['label'] = 'POPIA'
combined_df = pd.concat([df, df2])
combined_df = combined_df.drop(columns=["URL"])

# Generate Embeddings for Training Data
def get_embedding(text):
    response = openai.Embedding.create(
        input=text,
        model="text-embedding-ada-002"
    )
    return response['data'][0]['embedding']

combined_df['embedding'] = combined_df['Content'].apply(get_embedding)

# Define a Function to Test a New Document
def classify_document(document_text, combined_df, threshold=0.5):
    document_embedding = get_embedding(document_text)

    # Calculate similarity with each provision
    similarities = combined_df['embedding'].apply(lambda x: cosine_similarity([x], [document_embedding])[0][0])

    # Get the best match above threshold
    match = combined_df[similarities > threshold]
    if not match.empty:
        return match['label'].values[0]  # Returns GDPR or POPIA
    else:
        return "No match found"


In [ ]:
text = '''Personal information may only be processed if—
the data subject or a competent person where the data subject is a child consents to the processing;
processing is necessary to carry out actions for the conclusion or performance of a contract to which the data subject is party;
processing complies with an obligation imposed by law on the responsible party;
processing protects a legitimate interest of the data subject;
processing is necessary for the proper performance of a public law duty by a public body; or
processing is necessary for pursuing the legitimate interests of the responsible party or of a third party to whom the information is supplied.
The responsible party bears the burden of proof for the data subject’s or competent person’s consent as referred to in subsection (1)(a).
The data subject or competent person may withdraw his, her or its consent, as referred to in subsection (1)(a), at any time: Provided that the lawfulness of the processing of personal information before such withdrawal or the processing of personal information in terms of subsection (1)(b) to (f) will not be affected.
A data subject may object, at any time, to the processing of personal information—
in terms of subsection (1)(d) to (f), in the prescribed manner, on reasonable grounds relating to his, her or its particular situation, unless legislation provides for such processing; or
for purposes of direct marketing other than direct marketing by means of unsolicited electronic communications as referred to in section 69.
If a data subject has objected to the processing of personal information in terms of subsection (3), the responsible party may no longer process the personal information.'''

In [ ]:
print(classify_document(text, combined_df))

GDPR


In [ ]:
compliant_articles = []
non_compliant_articles = []
document_text = '''What Personal Information About Customers Does Amazon Collect?
 We collect your personal information in order to provide and continually improve our products and services.

 Here are the types of personal information we collect:

   Information You Give Us: We receive and store any information you provide in relation to Amazon Services. Click here to see examples of what we collect. You can choose not to provide certain information, but then you might not be able to take advantage of many of our Amazon Services.

  Automatic Information: We automatically collect and store certain types of information about your use of Amazon Services, including information about your interaction with content and services available through Amazon Services. Like many websites, we use cookies and other unique identifiers, and we obtain certain types of information when your web browser or device accesses Amazon Services and other content served by or on behalf of Amazon on other websites. Click here to see examples of what we collect.

   Information from Other Sources: We might receive information about you from other sources, such as updated delivery and address information from our carriers, which we use to correct our records and deliver your next purchase more easily. Click here to see additional examples of the information we receive

Back to Top

 For What Purposes Does Amazon Use Your Personal Information?
We use your personal information to operate, provide, develop, and improve the products and services that we offer our customers. These purposes include:

Purchase and delivery of products and services. We use your personal information to take and fulfill orders, deliver products and services, process payments, and communicate with you about orders, products and services, and promotional offers.
Provide, troubleshoot, and improve Amazon Services. We use your personal information to provide functionality, analyze performance, fix errors, and improve the usability and effectiveness of the Amazon Services.
Recommendations and personalization. We use your personal information to recommend features, products, and services that might be of interest to you, identify your preferences, and personalize your experience with Amazon Services.
Provide voice, image and camera services. When you use our voice, image and camera services, we use your voice input, images, videos, and other personal information to respond to your requests, provide the requested service to you, and improve our services. For more information about Alexa voice services, click here.
Comply with legal obligations. In certain cases, we collect and use your personal information to comply with laws. For instance, we collect from sellers information regarding place of establishment and bank account information for identity verification and other purposes.
Communicate with you. We use your personal information to communicate with you in relation to Amazon Services via different channels (e.g., by phone, e-mail, chat).
Advertising. We use your personal information to display interest-based ads for features, products, and services that might be of interest to you. We do not use information that personally identifies you to display interest-based ads. To learn more, please read our Interest-Based Ads notice.
Fraud Prevention and Credit Risks. We use personal information to prevent and detect fraud and abuse in order to protect the security of our customers, Amazon, and others. We may also use scoring methods to assess and manage credit risks.


 What About Cookies and Other Identifiers?
To enable our systems to recognize your browser or device and to provide and improve Amazon Services, we use cookies and other identifiers. For more information about cookies and how we use them, please read our Cookies Notice.
Back to Top

 Does Amazon Share Your Personal Information?
Information about our customers is an important part of our business and we are not in the business of selling our customers’ personal information to others. We share customers’ personal information only as described below and with Amazon.com, Inc. and subsidiaries that Amazon.com, Inc. controls that either are subject to this Privacy Notice or follow practices at least as protective as those described in this Privacy Notice.

Transactions involving Third Parties: We make available to you services, products, applications, or skills provided by third parties for use on or through Amazon Services. For example, the products you order through our marketplace are from third parties, you can download applications from third-party application providers from our App Store, and enable third-party skills through our Alexa services. We also offer services or sell product lines jointly with third-party businesses, such as sellers on the marketplace, restaurants registered on Amazon.in, merchants providing mobile recharges and bill-payment assistance. You can tell when a third party is involved in your transactions, and we share customers’ personal information related to those transactions with that third party.
Third-Party Service Providers: We employ other companies and individuals to perform functions on our behalf. Examples include fulfilling orders for products or services, delivering packages, sending postal mail and e-mail, removing repetitive information from customer lists, analyzing data, providing marketing assistance, providing search results and links (including paid listings and links), processing payments, transmitting content, scoring, assessing and managing credit risk, and providing customer service. These third-party service providers have access to personal information needed to perform their functions, but may not use it for other purposes. Further, they must process the personal information in accordance with applicable law.
Business Transfers: As we continue to develop our business, we might sell or buy other businesses or services. In such transactions, customer information generally is one of the transferred business assets but remains subject to the promises made in any pre-existing Privacy Notice (unless, of course, the customer consents otherwise). Also, in the unlikely event that Amazon.com, Inc. or Amazon Seller Services Private Limited or any of its affiliates, or substantially all of their assets are acquired, customer information will of course be one of the transferred assets.
Protection of Amazon and Others: We release account and other personal information when we believe release is appropriate to comply with the law; enforce or apply our Conditions of Use and other agreements; or protect the rights, property, or safety of Amazon, our users, or others. This includes exchanging information with other companies and organizations for fraud protection and credit risk reduction.
Other than as set out above, you will receive notice when personal information about you might be shared with third parties, and you will have an opportunity to choose not to share the information.

Back to Top

 How Secure Is Information About Me?
 We design our systems with your security and privacy in mind.

We work to protect the security of your personal information during transmission by using encryption protocols and software.
 We follow the Payment Card Industry Data Security Standard (PCI DSS) when handling payment card data.
 We maintain physical, electronic, and procedural safeguards in connection with the collection, storage, processing, and disclosure of personal customer information. Our security procedures mean that we may occasionally request proof of identity before we disclose personal information to you.
 Our devices offer security features to protect them against unauthorized access and loss of data. You can control these features and configure them based on your needs. Click here for more information on how to manage the security settings of your device.
 It is important for you to protect against unauthorized access to your password and to your computers, devices and applications. Be sure to sign off when finished using a shared computer. Click here for more information on how to sign off.
Back to Top

 What About Advertising?
  Third-Party Advertisers and Links to Other Websites: Amazon Services may include third-party advertising and links to other websites and apps. Third-party advertising partners may collect information about you when you interact with their content, advertising, and services. For more information about third-party advertising at Amazon, including interest-based ads, please read our Interest-Based Ads policy. To adjust your advertising preferences, please go to the Advertising Preferences page.

Use of Third-Party Advertising Services: We provide ad companies with information that allows them to serve you with more useful and relevant Amazon ads and to measure their effectiveness. We never share your name or other information that directly identifies you when we do this. Instead, we use an advertising identifier like a cookie, a device identifier, or a code derived from applying irreversible cryptography to other information like an email address. For example, if you have already downloaded one of our apps, we will share your advertising identifier and data about that event so that you will not be served an ad to download the app again. Some ad companies also use this information to serve you relevant ads from other advertisers. You can learn more about how to opt-out of interest-based advertising by going to the Advertising Preferences page.

Back to Top

 What Information Can I Access?
 You can access your information, including your name, address, payment options, profile information, Prime membership, household settings, and purchase history in the "Your Account" section of the website or mobile application. Click here for a list of examples that you can access. To request access to personal information that is not available through Your Account you can submit a request here.

Back to Top

 What Choices Do I Have?
If you have any questions as to how we collect and use your personal information, please contact our Grievance Officer. Many of our Amazon Services also include settings that provide you with options as to how your information is being used

 As described above, you can always choose not to provide certain information, but then you might not be able to take advantage of many of the Amazon Services.
You can add or update certain information on pages such as those referenced in What Information Can I Access?. When you update information, we usually keep a copy of the prior version for our records
If you do not want to receive e-mail or other communications from us, please adjust your Customer Communication Preferences. If you don’t want to receive in-app notifications from us, please adjust your notification settings in the app or device
If you do not want to see interest-based ads, please adjust your Advertising Preferences.
The Help feature on most browsers and devices will tell you how to prevent your browser or device from accepting new cookies or other identifiers, how to have the browser notify you when you receive a new cookie or how to block cookies altogether. Because cookies and identifiers allow you to take advantage of some essential features of Amazon Services, we recommend that you leave them turned on. For instance, if you block or otherwise reject our cookies, you will not be able to add items to your Shopping Cart, proceed to Checkout, or use any Services that require you to Sign in. For more information about cookies and other identifiers, see our Cookies Notice.
If you want to browse our websites without linking the browsing history to your account, you may do so by logging out of your account here and blocking cookies on your browser.
You will also be able to opt out of certain other types of data usage by updating your settings on the applicable Amazon website (e.g., in "Manage Your Content and Devices"), device, or application. For more information click here. Most non-Amazon devices also provide users with the ability to change device permissions (e.g., disable/access location services, contacts). For most devices, these controls are located in the device's settings menu. If you have questions about how to change your device permissions on devices manufactured by third parties, we recommend you contact your mobile service carrier or your device manufacturer.
 If you are a seller, you can add or update certain information in Seller Central , update your account information by accessing your Seller Account Information, and adjust your e-mail or other communications you receive from us by updating your Notification Preferences.
Back to Top

 Are Children Allowed to Use Amazon Services ?
 Amazon does not sell products for purchase by children. We sell children's products for purchase by adults. If you are under the age of 18 years, you may use Amazon Services only with the involvement of a parent or guardian.

Back to Top

 Conditions of Use, Notices , and Revisions
 If you choose to use Amazon Services, your use and any dispute over privacy is subject to this Notice and our Conditions of Use, including limitations on damages, resolution of disputes, and application of the prevailing law in India. If you have any concern about privacy at Amazon, please contact us with a thorough description, and we will try to resolve it. Our business changes constantly, and our Privacy Notice will change also. You should check our websites frequently to see recent changes.

Unless stated otherwise, our current Privacy Notice applies to all information that we have about you and your account. We stand behind the promises we make, however, and will never materially change our policies and practices to make them less protective of customer information collected in the past without the consent of affected customers.

Back to Top

 Related Practices and Information
Conditions of Use
Seller Program Policies
Help Department
Most Recent Purchases
Your Profile and Community Guidelines
 Examples of Information Collected
   Information You Give Us When You Use Amazon Services

You provide information to us when you:

search or shop for products or services in our marketplace;
 add or remove an item from your cart, or place an order through or use Amazon Services;
 download, stream, view, or use content on a device or through a service or application on a device;
 provide information in Your Account (and you might have more than one if you have used more than one e-mail address or mobile number when shopping with us) or Your Profile;
 talk to or otherwise interact with our Alexa Voice service;
 upload your contacts, including access to mobile device contacts for certain services;
 configure your settings on, provide data access permissions for, or interact with an Amazon device or service;
 provide information in your Seller Account, Restaurant Central Account, Service Provider Account, or any other account we make available that allows you to develop or offer software, goods, or services to Amazon customers;
 offer your products or services on or through Amazon Services;
 communicate with us by phone, e-mail, or otherwise;
 complete a questionnaire, a support ticket, or a contest entry form;
 upload or stream images, videos or audio or other files while using Amazon Services;
 use our services such as Prime Video;
 compile Playlists, Watchlists, Wish Lists or other gift registries;
 participate in Discussion Boards or other community features;
 provide and rate Reviews;
 specify a Special Occasion Reminder; or
 employ Product Availability Alerts, such as Available to Order Notifications.


As a result of those actions, you might supply us with such information as:

 identifying information such as your name, address and phone numbers;
 payment information;
 your age;
 your location information;
your IP address;
people, addresses and phone numbers listed in your Addresses;
e-mail addresses of your friends and other people;
content of reviews and e-mails to us;
the personal description and photograph in Your Profile;
voice recordings when you speak to Alexa;
images, videos and other content collected or stored in connection with Amazon Services;
information and officially valid documents regarding identity and address information, including PAN numbers;
credit history information;
corporate and financial information; and
device log files and configurations, including Wi-Fi credentials, if you choose to automatically synchronize them with your other Amazon devices.
   Automatic Information

 Examples of the information we collect and analyze include:

 the internet protocol (IP) address used to connect your computer to the internet;
 login, e-mail address, and password;
 the location of your device or computer;
 content interaction information, such as content downloads, streams, and playback details, including duration and number of simultaneous streams and downloads, and network details for streaming and download quality, including information about your internet service provider;
 device metrics such as when a device is in use, application usage, connectivity data, and any errors or event failures;
 Amazon Services metrics (e.g., the occurrences of technical errors, your interactions with service features and content, your settings preferences and backup information, location of your device running an application, information about uploaded images and files such as the file name, dates, times and location of your images);
 version and time zone settings;
 purchase and content use history, which we sometimes aggregate with similar information from other customers to create features like Amazon Bestsellers;
 the full Uniform Resource Locator (URL) clickstream to, through, and from our websites, including date and time; products and content you viewed or searched for; page response times, download errors, length of visits to certain pages, and page interaction information (such as scrolling, clicks, and mouse-overs);
 phone numbers used to call our customer service number; and
 images or videos when you shop in our marketplace using Amazon Services.
We may also use device identifiers, cookies, and other technologies on devices, applications, and our web pages to collect browsing, usage, or other technical information.

   Information from Other Sources

Examples of information we receive from other sources include:

updated delivery and address information from our carriers or other third parties, which we use to correct our records and deliver your next purchase or communication more easily;
 account information, purchase or redemption information and page-view information from some merchants with which we operate co-branded businesses or for which we provide technical, fulfillment, advertising or other services;
 information about your interactions with products and services offered by our affiliates;
 search results and links, including paid listings (such as Sponsored Links);
 information about internet-connected devices and services linked with Alexa; and
 credit history information from credit bureaus, which we use to help prevent and detect fraud and to offer certain credit or financial services to some customers.
   Information You Can Access

Examples of information you can access through Amazon Services include:

 status of recent orders (including subscriptions);
 your complete order history;
 personally identifiable information (including name, e-mail, password, and address book);
 payment settings (including payment method information, promotional certificate, and gift card balances and 1-Click settings);
 e-mail notification settings (including Product Availability Alerts, Delivers, Special Occasion Reminders and newsletters);
 recommendations and the products you recently viewed that are the basis for recommendations (including Recommended for You and Improve Your Recommendations);
 shopping lists and gift registries (including Wish Lists);
 your content, devices, services, and related settings, and communications and personalized advertising preferences;
 content that you recently viewed;
 voice recordings associated with your account;
 Your Profile (including your product Reviews, Recommendations, Reminders and personal profile);
 If you are a seller, you can access your account and other information, and adjust your communications preferences, by updating your account in Seller Central .
 If you are a restaurant, you can access your account and other information, and adjust your communication preferences, by updating your account in Restaurant Central;
 If you are a service provider listing on our Service Provider Network, you can access your account and other information, and adjust your communication preferences, by updating your account in Service Provider Central;
'''

for article in preprocessed_articles:
    # Check if the document is compliant with the requirement for this article
    compliance_result = check_compliance(document_text, article["requirement"])

    # Categorize articles based on compliance status
    if "Compliant" in compliance_result:
        compliant_articles.append(article)
    else:
        article["reason"] = compliance_result
        non_compliant_articles.append(article)

# Output the results
print("Compliant Articles:")
for item in compliant_articles:
    print(f"Article {item['article']}:")
    print(f"General Summary: {item['general_summary']}")
    print(f"Requirement: {item['requirement']}")

print("\nNon-Compliant Articles:")
for item in non_compliant_articles:
    print(f"Article {item['article']}:")
    print(f"General Summary: {item['general_summary']}")
    print(f"Requirement: {item['requirement']}")
    print(f"Reason: {item['reason']}")

Compliant Articles:
Article 28:
General Summary: General Summary:
This article outlines the responsibilities and requirements for processors who handle data on behalf of a controller under the GDPR. It emphasizes the need for processors to have appropriate technical and organizational measures in place to protect data subjects' rights and ensure compliance with the regulation.
Requirement: - Processors must use only processors that provide sufficient guarantees to meet GDPR requirements and protect data subjects' rights.
- Processors cannot engage another processor without the controller's authorization.
- Processing by a processor must be governed by a contract or legal act that outlines specific details such as the nature of processing, data subjects, and obligations.
- Processors must follow documented instructions from the controller, ensure confidentiality, take necessary security measures, assist the controller in responding to data subject requests, and assist in compliance with

In [ ]:
with open("gpdr_pr.csv", mode="w", newline="") as csv_file:
    # Change fieldnames to match the keys in the article dictionary
    fieldnames = ["article", "general_summary", "requirement"]
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)

    writer.writeheader()
    for article in preprocessed_articles:
        writer.writerow(article)

print(f"Preprocessed articles have been saved to gpdr_pr.csv") # Changed output_csv_file_path to the actual filename

Preprocessed articles have been saved to gpdr_pr.csv


In [ ]:
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import pandas as pd

# Step 1: Load and summarize articles
data = pd.read_csv('final_regulatory_preprocessed_papers_popi.csv')
data = data[10:30]
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
data['Summary'] = data['Content'].apply(lambda x: summarizer(x[:1024], max_length=100, min_length=30, do_sample=False)[0]['summary_text'])

# Step 2: Generate embeddings for each article summary
model = SentenceTransformer('all-MiniLM-L6-v2')
data['Embedding'] = data['Summary'].apply(lambda x: model.encode(x))

# Step 3: Summarize and embed the document
document = """Google GDPR Privacy Policy
1. Introduction
Purpose: Google is committed to safeguarding personal data and ensuring that privacy is protected by design and by default. This privacy policy outlines how Google collects, uses, processes, and shares personal information, in compliance with the EU General Data Protection Regulation (GDPR).

Scope: This policy applies to all data processing activities performed by Google in relation to personal data of users located in the European Union (EU) and European Economic Area (EEA).

2. Definitions
Personal Data: Information that relates to an identified or identifiable individual.
Processing: Any operation performed on personal data, including collection, storage, and use.
Data Subject: Any individual whose personal data is being processed by Google.
Data Controller: Google, Inc., responsible for determining the purposes and means of processing personal data.
Consent: Freely given, specific, informed, and unambiguous indication of the data subject's wishes by which they agree to the processing of personal data.
3. Data Protection Principles (Articles 5-6)
Google adheres to the following GDPR data protection principles:

Lawfulness, Fairness, and Transparency: Google processes personal data lawfully, fairly, and in a transparent manner.
Purpose Limitation: Data is collected for specified, explicit, and legitimate purposes and not further processed in a manner incompatible with those purposes.
Data Minimization: Google only collects data that is necessary for each purpose.
Accuracy: Google strives to keep personal data accurate and up-to-date.
Storage Limitation: Personal data is stored only as long as necessary for the purposes for which it was collected.
Integrity and Confidentiality: Google secures data to protect against unauthorized processing, accidental loss, and damage.
Accountability: Google is responsible for ensuring compliance with GDPR and these principles.
4. Lawful Bases for Processing (Articles 6-7)
Google processes personal data based on one or more lawful grounds:

Consent: When required, Google will obtain consent before collecting or processing personal data.
Contractual Necessity: Data processing may be necessary to fulfill contractual obligations with users.
Legal Obligation: Google may process data to comply with applicable laws and regulations.
Legitimate Interests: Data processing may be justified by Google’s legitimate interests, provided those interests are not overridden by the rights and interests of users.
5. Data Subject Rights (Articles 12-23)
Google recognizes and upholds the following rights of data subjects:

Right to Access (Article 15): Users may request access to their personal data and information on how Google processes it.
Right to Rectification (Article 16): Users may request correction of inaccurate personal data.
Right to Erasure (Article 17): Users have the right to request deletion of personal data under specific conditions.
Right to Restriction of Processing (Article 18): Users may request restriction of data processing in certain circumstances.
Right to Data Portability (Article 20): Users can request a copy of their data in a structured, commonly used, and machine-readable format.
Right to Object (Article 21): Users have the right to object to certain types of data processing, including direct marketing.
Rights Related to Automated Decision-Making (Article 22): Users have the right not to be subject to decisions based solely on automated processing, where applicable.
6. Consent Management (Articles 7-8)
Obtaining Consent: Google provides clear, accessible methods for users to provide consent, ensuring it is specific, informed, and unambiguous.
Withdrawing Consent: Users can withdraw consent at any time through account settings or by contacting Google’s support team.
7. Data Protection Impact Assessments (DPIA) (Article 35)
Google conducts Data Protection Impact Assessments for processing activities likely to result in a high risk to data subjects’ privacy, such as large-scale processing of sensitive data.

8. Data Breach Notification (Articles 33-34)
Internal Reporting: Google has an internal data breach response policy to detect, report, and investigate any data breach incidents.
Notification to Supervisory Authority: In the event of a personal data breach, Google will notify the relevant supervisory authority within 72 hours, if required.
Notification to Data Subjects: If a data breach poses a high risk to users' rights and freedoms, Google will inform the affected data subjects without undue delay.
9. Data Transfers (Articles 44-50)
International Transfers: Google transfers data outside the EU only when adequate protections are in place, such as using EU Standard Contractual Clauses.
Third-Party Service Providers: Google performs due diligence on third-party processors to ensure they comply with GDPR standards when processing data internationally.
10. Data Security (Article 32)
Organizational Measures: Google employs organizational policies to secure personal data, including data protection training for employees.
Technical Measures: Technical safeguards include encryption, pseudonymization, and multi-factor authentication to prevent unauthorized access to personal data.
11. Accountability and Governance (Articles 24-31)
Data Protection Officer (DPO): Google has appointed a DPO to oversee GDPR compliance. Users can contact the DPO at [contact details].
Documentation of Processing Activities: Google maintains records of processing activities to ensure accountability.
Employee Training and Awareness: Google provides training for employees on data protection responsibilities.
12. Audits and Compliance Checks
Google conducts regular audits and compliance checks to maintain adherence to GDPR and ensure continuous improvement in data protection practices.

13. Updates to This Policy
Policy Review and Update Schedule: This policy is reviewed regularly and updated as necessary. Any significant changes will be communicated to users through Google’s standard notification methods.

14. Contact Information
For questions, requests, or concerns about this policy or GDPR compliance, please contact Google’s Data Protection Officer at:
[DPO Contact Details]15. Additional Appendices
Appendix A: Summary of GDPR Articles covered.
Appendix B: List of Data Processing Activities.
Appendix C: Sample Consent Forms and Notices.""" # Replace with the document string
document_summary = summarizer(document[:1024], max_length=100, min_length=30, do_sample=False)[0]['summary_text']
document_embedding = model.encode(document_summary)

# Step 4: Compare document with each article for compliance
threshold = 0.8  # Set threshold for compliance
compliant_articles = []

for i, article_emb in enumerate(data['Embedding']):
    similarity = util.cos_sim(document_embedding, article_emb)[0][0].item()
    if similarity >= threshold:
        compliant_articles.append(data.iloc[i]['Article Name'])

# Output the list of compliant articles
print("The document complies with the following articles:", compliant_articles)



/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
Your max_length is set to 100, but your input_length is only 70. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=35)
Your max_length is set to 100, but your input_length is only 63. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g.

The document complies with the following articles: []


In [ ]:
import requests
from bs4 import BeautifulSoup
from transformers import DistilBertTokenizer, DistilBertModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import re

# Example compliance requirements for various regulations (GDPR, PCI DSS, HIPAA)
data=pd.read_csv("final_regulatory_preprocessed_papers_popi.csv")
compliance_requirements = data['Content']

# Keywords to check manually in the privacy policy text
manual_keywords = [
    "user consent", "data collection", "data deletion", "encryption", "cardholder data",
    "health information", "authorized personnel", "withdraw consent", "data retention",
    "data breach", "rectification"
]

# Function to fetch privacy policy text from a URL
def fetch_privacy_policy_from_url(url):
    response = requests.get(url)
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        # Extract all text paragraphs from the page
        paragraphs = soup.find_all('p','li')
        policy_text = " ".join([p.get_text() for p in paragraphs])
        return policy_text
    else:
        raise Exception("Failed to retrieve the privacy policy page.")

# Function to clean the text (remove unnecessary elements)
def clean_text(text):
    # Remove special characters, extra spaces, and newlines
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with one
    text = re.sub(r'\n+', ' ', text)  # Remove newlines
    text = re.sub(r'\[.*?\]', '', text)  # Remove anything inside square brackets
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    return text.strip()

# Function to encode text using DistilBERT and return embeddings
def get_bert_embeddings(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze()

# Function to check compliance based on semantic similarity using BERT
def check_privacy_policy_compliance(privacy_policy_text, compliance_requirements, model, tokenizer, threshold=0.85):
    # Clean the privacy policy text
    privacy_policy_text = clean_text(privacy_policy_text)

    # Get embeddings for the privacy policy text
    policy_embedding = get_bert_embeddings(privacy_policy_text, model, tokenizer)

    compliance_check_results = {}

    # Check each compliance requirement
    for requirement in compliance_requirements:
        requirement_embedding = get_bert_embeddings(requirement, model, tokenizer)

        # Compute cosine similarity between the privacy policy and the compliance requirement
        similarity = cosine_similarity(policy_embedding.unsqueeze(0), requirement_embedding.unsqueeze(0))

        # If similarity is above the threshold, consider it compliant
        compliance_check_results[requirement] = similarity[0][0] > threshold

    return compliance_check_results

# Function to check if manual keywords are present in the policy text
def check_manual_keywords(privacy_policy_text, keywords):
    keyword_results = {}
    for keyword in keywords:
        keyword_results[keyword] = keyword.lower() in privacy_policy_text.lower()
    return keyword_results

# Load DistilBERT model and tokenizer
model_name = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
model = DistilBertModel.from_pretrained(model_name)

# URL of the privacy policy (Slack in this case)
url = "https://policies.google.com/privacy?hl=en-US"

# Fetch privacy policy text
try:
    privacy_policy_text = fetch_privacy_policy_from_url(url)
    privacy_policy_text ="""Google GDPR Privacy Policy
1. Introduction
Purpose: Google is committed to safeguarding personal data and ensuring that privacy is protected by design and by default. This privacy policy outlines how Google collects, uses, processes, and shares personal information, in compliance with the EU General Data Protection Regulation (GDPR).

Scope: This policy applies to all data processing activities performed by Google in relation to personal data of users located in the European Union (EU) and European Economic Area (EEA).

2. Definitions
Personal Data: Information that relates to an identified or identifiable individual.
Processing: Any operation performed on personal data, including collection, storage, and use.
Data Subject: Any individual whose personal data is being processed by Google.
Data Controller: Google, Inc., responsible for determining the purposes and means of processing personal data.
Consent: Freely given, specific, informed, and unambiguous indication of the data subject's wishes by which they agree to the processing of personal data.
3. Data Protection Principles (Articles 5-6)
Google adheres to the following GDPR data protection principles:

Lawfulness, Fairness, and Transparency: Google processes personal data lawfully, fairly, and in a transparent manner.
Purpose Limitation: Data is collected for specified, explicit, and legitimate purposes and not further processed in a manner incompatible with those purposes.
Data Minimization: Google only collects data that is necessary for each purpose.
Accuracy: Google strives to keep personal data accurate and up-to-date.
Storage Limitation: Personal data is stored only as long as necessary for the purposes for which it was collected.
Integrity and Confidentiality: Google secures data to protect against unauthorized processing, accidental loss, and damage.
Accountability: Google is responsible for ensuring compliance with GDPR and these principles.
4. Lawful Bases for Processing (Articles 6-7)
Google processes personal data based on one or more lawful grounds:

Consent: When required, Google will obtain consent before collecting or processing personal data.
Contractual Necessity: Data processing may be necessary to fulfill contractual obligations with users.
Legal Obligation: Google may process data to comply with applicable laws and regulations.
Legitimate Interests: Data processing may be justified by Google’s legitimate interests, provided those interests are not overridden by the rights and interests of users.
5. Data Subject Rights (Articles 12-23)
Google recognizes and upholds the following rights of data subjects:

Right to Access (Article 15): Users may request access to their personal data and information on how Google processes it.
Right to Rectification (Article 16): Users may request correction of inaccurate personal data.
Right to Erasure (Article 17): Users have the right to request deletion of personal data under specific conditions.
Right to Restriction of Processing (Article 18): Users may request restriction of data processing in certain circumstances.
Right to Data Portability (Article 20): Users can request a copy of their data in a structured, commonly used, and machine-readable format.
Right to Object (Article 21): Users have the right to object to certain types of data processing, including direct marketing.
Rights Related to Automated Decision-Making (Article 22): Users have the right not to be subject to decisions based solely on automated processing, where applicable.
6. Consent Management (Articles 7-8)
Obtaining Consent: Google provides clear, accessible methods for users to provide consent, ensuring it is specific, informed, and unambiguous.
Withdrawing Consent: Users can withdraw consent at any time through account settings or by contacting Google’s support team.
7. Data Protection Impact Assessments (DPIA) (Article 35)
Google conducts Data Protection Impact Assessments for processing activities likely to result in a high risk to data subjects’ privacy, such as large-scale processing of sensitive data.

8. Data Breach Notification (Articles 33-34)
Internal Reporting: Google has an internal data breach response policy to detect, report, and investigate any data breach incidents.
Notification to Supervisory Authority: In the event of a personal data breach, Google will notify the relevant supervisory authority within 72 hours, if required.
Notification to Data Subjects: If a data breach poses a high risk to users' rights and freedoms, Google will inform the affected data subjects without undue delay.
9. Data Transfers (Articles 44-50)
International Transfers: Google transfers data outside the EU only when adequate protections are in place, such as using EU Standard Contractual Clauses.
Third-Party Service Providers: Google performs due diligence on third-party processors to ensure they comply with GDPR standards when processing data internationally.
10. Data Security (Article 32)
Organizational Measures: Google employs organizational policies to secure personal data, including data protection training for employees.
Technical Measures: Technical safeguards include encryption, pseudonymization, and multi-factor authentication to prevent unauthorized access to personal data.
11. Accountability and Governance (Articles 24-31)
Data Protection Officer (DPO): Google has appointed a DPO to oversee GDPR compliance. Users can contact the DPO at [contact details].
Documentation of Processing Activities: Google maintains records of processing activities to ensure accountability.
Employee Training and Awareness: Google provides training for employees on data protection responsibilities.
12. Audits and Compliance Checks
Google conducts regular audits and compliance checks to maintain adherence to GDPR and ensure continuous improvement in data protection practices.

13. Updates to This Policy
Policy Review and Update Schedule: This policy is reviewed regularly and updated as necessary. Any significant changes will be communicated to users through Google’s standard notification methods.

14. Contact Information
For questions, requests, or concerns about this policy or GDPR compliance, please contact Google’s Data Protection Officer at:
[DPO Contact Details]15. Additional Appendices
Appendix A: Summary of GDPR Articles covered.
Appendix B: List of Data Processing Activities.
Appendix C: Sample Consent Forms and Notices."""
    print("Privacy Policy Text Fetched Successfully!\n")

    # Check compliance of the fetched policy with BERT
    compliance_results_bert = check_privacy_policy_compliance(privacy_policy_text, compliance_requirements, model, tokenizer, threshold=0.7)

    # Check manual keyword compliance
    compliance_results_keywords = check_manual_keywords(privacy_policy_text, manual_keywords)

    # Print the compliance results from BERT
    print("Compliance Check Results (BERT-based):")
    for requirement, is_followed in compliance_results_bert.items():
        print(f"{requirement}: {'Compliant' if is_followed else 'Not Compliant'}")

    # Print the results from manual keyword matching
    print("\nCompliance Check Results (Manual Keywords):")
    for keyword, is_present in compliance_results_keywords.items():
        print(f"Keyword '{keyword}': {'Present' if is_present else 'Not Present'}")

except Exception as e:
    print(f"Error: {str(e)}")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Privacy Policy Text Fetched Successfully!

Compliance Check Results (BERT-based):
In this Act, unless the context indicates otherwise —
‘‘biometrics’’ means a technique of personal identification that is based on physical, physiological or behavioural characterisation including blood typing, fingerprinting, DNA analysis, retinal scanning and voice recognition;‘‘child’’ means a natural person under the age of 18 years who is not legally competent, without the assistance of a competent person, to take any action or decision in respect of any matter concerning him- or herself;‘‘code of conduct’’ means acode of conductissued in terms of Chapter 7;‘‘competent person’’ means any person who is legally competent to consent to any action or decision being taken in respect of any matter concerning a child;‘‘consent’’ means any voluntary, specific and informed expression of will in terms of which permission is given for the processing of personal information;‘‘Constitution’’ means the Constitutio

In [ ]:
import csv
import openai

# Load the OpenAI API key
openai.api_key = 'YOUR_API_KEY'

# Define a function to generate both the general summary and requirements for each GDPR article
def process_gdpr_article(article_text):
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that summarizes GDPR articles."},
            {"role": "user", "content": f"Summarize the following GDPR article into two sections:\n\n1. General Summary: Provide a high-level summary of this article.\n2. Requirement: List specific compliance requirements based on this article.\n\nArticle:\n{article_text}\n\nGeneral Summary:"}
        ],
        max_tokens=250,
        temperature=0.3,
    )
    # Extract response text from the chat model's response
    response_text = response['choices'][0]['message']['content'].strip()
    try:
        # Split response into general summary and requirement
        general_summary, requirement = response_text.split("Requirement:", 1)
    except ValueError:
        general_summary, requirement = response_text, "Requirement not extracted properly."

    return general_summary.strip(), requirement.strip()

# Define a function to check compliance of a document against the GDPR requirements
def check_compliance(document_text, requirement_summary):
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that checks GDPR compliance."},
            {"role": "user", "content": f"Check if the following document is compliant with this GDPR requirement:\n\nRequirement: {requirement_summary}\n\nDocument:\n{document_text}\n\nCompliance status (Compliant/Non-compliant) and reasons:"}
        ],
        max_tokens=150,
        temperature=0.3,
    )
    return response['choices'][0]['message']['content'].strip()

# Load and preprocess the CSV data
csv_file_path = "gdpr_articles.csv"
articles = []
preprocessed_articles = []

with open(csv_file_path, mode="r") as csv_file:
    reader = csv.DictReader(csv_file)
    for row in reader:
        # Process the article to get general summary and requirement
        general_summary, requirement_summary = process_gdpr_article(row["Content"])
        preprocessed_articles.append({
            "article": row["Article Name"],
            "general_summary": general_summary,
            "requirement": requirement_summary
        })


